# Violin Plots

One last visualization tool, widely used in data science: the **violin plot**. Its trick is simple — it contains the information of *both* the kernel density estimate and the box plot: the KDE curve (mirrored to form the violin body) drawn alongside the median, quartiles, and whiskers. Shape and summary statistics in a single figure. It is based on the fifteenth video of Lesson 3 (Describing Distributions).

## 1. Introduction

The last three notebooks each built one visualization: quantiles gave the numbers, the **box plot** turned five of them into a compact picture, and the **KDE** recovered the whole smooth shape. Each has a weakness: the box plot hides shape (bimodality is invisible), and a bare KDE curve carries no summary statistics.

The violin plot fuses the two. All you do is put the kernel density estimate at the side of the box plot — traditionally mirrored left and right, so the density becomes the "body" of a violin, with the quartile information drawn inside.

After completing this notebook you will be able to:

- Read every element of a violin plot: KDE outline, median/mean, quartiles, whiskers.
- Build violin plots with matplotlib and enrich them with quartile markers.
- Choose between box, KDE, and violin plots — and know when the violin's extra ink pays off (especially for comparing groups and spotting multimodality).

## 2. Theory

### The construction

A violin plot for one dataset:

1. Compute the **kernel density estimate** of the data (previous notebook).
2. Draw the KDE curve vertically and **mirror it** on the other side — the width of the resulting body at each height is proportional to the estimated density there. Wide sections = many observations; narrow sections = few.
3. Inside the body, mark the **box-plot statistics**: the median (and often the mean), the quartiles $Q_1$ and $Q_3$, and the whiskers with their 1.5·IQR rule.

So one figure answers both questions at once: *what is the shape of the distribution?* (the violin outline) and *where are the center, the middle 50%, and the outliers?* (the inner marks).

### Reading a violin plot

- **Fat regions** are where the data concentrates — a violin with two bulges is a bimodal dataset that a box plot would silently flatten.
- **Asymmetry** of the body (bulge low, long thin neck upward) is skew, matching an off-center median in the box.
- Side-by-side violins compare groups: differences in center, spread, *and* shape are visible simultaneously.

### Caveats

The violin inherits the KDE's bandwidth choice: too small and the body grows spurious wiggles, too large and real features blur. And near hard data boundaries (e.g. budgets can't be negative) the KDE smears a little probability past the edge. As always: the plot is an *estimate*, not the data itself.

## 3. Worked Example: the newspaper budgets as a violin

For the 12-point newspaper sample (five-number summary $8.7, 18.35, 27.8, 52.95, 75$; right-skewed):

- the body is **widest around `$18–30k`**, where seven of the twelve budgets cluster;
- a **long, thinner upper section** stretches toward `$75k` — the right skew, visible as shape;
- inside, the median line sits at 27.8, low within the box $[18.35, 52.95]$ — the same skew, visible as statistics.

The violin shows in one glance what previously took a box plot *plus* a KDE.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

newspaper = np.array([8.7, 14.2, 18.3, 18.4, 23.2, 25.9,
                      29.7, 35.2, 51.2, 54.7, 65.9, 75.0])

### A basic violin plot

`plt.violinplot` draws the mirrored KDE; we ask it to mark the median and extremes too.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
parts = ax.violinplot(newspaper, showmedians=True, showextrema=True)
parts["bodies"][0].set_facecolor("tab:blue")
parts["bodies"][0].set_alpha(0.5)
ax.set_title("Violin plot of the 12 newspaper budgets")
ax.set_ylabel("budget ($1000s)")
ax.set_xticks([])
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

### The full anatomy: violin + box + points

The video's labeled picture: KDE outline, quartile box, median, mean, and whiskers — with the raw data jittered on top for honesty.

In [ ]:
q1, q2, q3 = np.quantile(newspaper, [0.25, 0.5, 0.75], method="midpoint")
mean = newspaper.mean()

fig, ax = plt.subplots(figsize=(7, 6))
parts = ax.violinplot(newspaper, positions=[0], showextrema=False, widths=0.9)
parts["bodies"][0].set_facecolor("tab:blue")
parts["bodies"][0].set_alpha(0.4)

# the box-plot skeleton inside the violin
ax.vlines(0, q1, q3, color="black", linewidth=8, alpha=0.8)          # the box
ax.vlines(0, newspaper.min(), newspaper.max(), color="black", lw=1)  # whiskers
ax.plot(0, q2, "o", color="white", markersize=9, zorder=3, label="median = 27.8")
ax.plot(0, mean, "D", color="tab:red", markersize=7, zorder=3,
        label=f"mean = {mean:.1f}")
jitter = rng.uniform(-0.06, 0.06, size=len(newspaper))
ax.plot(jitter + 0.28, newspaper, "o", color="tab:orange", alpha=0.8,
        label="data points")

ax.annotate("KDE outline\n(mirrored)", xy=(-0.33, 22), fontsize=10, ha="center")
ax.annotate("Q3 = 52.95", xy=(0.06, q3), fontsize=9)
ax.annotate("Q1 = 18.35", xy=(0.06, q1), fontsize=9)
ax.set_title("Anatomy of a violin plot: KDE shape + box-plot statistics")
ax.set_ylabel("budget ($1000s)")
ax.set_xticks([])
ax.set_xlim(-0.7, 0.7)
ax.grid(True, alpha=0.3, axis="y")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### Comparing groups: three ad channels side by side

The violin's favorite job — center, spread, and shape of several groups in one look.

In [ ]:
tv = np.array([7.8, 66.1, 44.5, 100.4, 151.5, 180.8,
               199.8, 214.7, 218.4, 228.3, 240.1, 293.6])
radio = np.array([2.6, 5.7, 8.6, 10.8, 16.9, 24.6,
                  26.7, 33.5, 36.9, 39.6, 41.3, 45.9])

fig, ax = plt.subplots(figsize=(9, 5))
parts = ax.violinplot([newspaper, tv, radio], showmedians=True)
for body, color in zip(parts["bodies"], ["tab:blue", "tab:green", "tab:purple"]):
    body.set_facecolor(color)
    body.set_alpha(0.5)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(["newspaper", "TV", "radio"])
ax.set_title("Ad budgets by channel: three violins, one comparison")
ax.set_ylabel("budget ($1000s)")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 5. Visualizations

### Box vs. KDE vs. violin on a bimodal dataset

The closing argument for violins: on two-cluster data the box plot sees nothing, the KDE sees the shape but no statistics, and the violin sees both.

In [ ]:
rng = np.random.default_rng(42)
bimodal = np.concatenate([rng.normal(30, 4, size=500), rng.normal(70, 4, size=500)])

def gaussian(x, center, sigma):
    return np.exp(-0.5 * ((x - center) / sigma)**2) / (sigma * np.sqrt(2 * np.pi))

def kde(x, data, sigma):
    return np.mean([gaussian(x, xi, sigma) for xi in data], axis=0)

grid = np.linspace(5, 95, 500)
sigma_b = 1.06 * bimodal.std() * len(bimodal)**(-1/5)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
axes[0].boxplot(bimodal, widths=0.5)
axes[0].set_title("box plot: clusters hidden")
axes[0].set_xticks([])

axes[1].plot(kde(grid, bimodal, sigma_b), grid, color="tab:red", linewidth=2)
axes[1].set_title("KDE: shape only")
axes[1].set_xlabel("density")

parts = axes[2].violinplot(bimodal, showmedians=True, widths=0.8)
parts["bodies"][0].set_facecolor("tab:blue")
parts["bodies"][0].set_alpha(0.5)
axes[2].set_title("violin: shape AND statistics")
axes[2].set_xticks([])

for ax in axes:
    ax.set_ylim(5, 95)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("value")
plt.suptitle("Why data scientists reach for violin plots")
plt.tight_layout()
plt.show()

### The bandwidth caveat carries over

Same data, three bandwidths: the violin's body is only as trustworthy as the KDE behind it.

In [ ]:
data = np.sort(newspaper)
grid = np.linspace(-10, 100, 500)

fig, axes = plt.subplots(1, 3, figsize=(12, 4.5), sharey=True)
for ax, s in zip(axes, [2, 6, 20]):
    density = kde(grid, data, s)
    scaled = density / density.max() * 0.4
    ax.fill_betweenx(grid, -scaled, scaled, color="tab:blue", alpha=0.5)
    ax.plot(np.zeros_like(data), data, "o", color="black", markersize=4)
    ax.set_title(f"bandwidth sigma = {s}")
    ax.set_xticks([])
    ax.grid(True, alpha=0.3, axis="y")
axes[0].set_ylabel("budget ($1000s)")
plt.suptitle("A violin is a mirrored KDE: bandwidth still matters")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** From the anatomy figure alone: which features of the newspaper violin tell you the data is right-skewed? Name one clue from the *shape* and one from the *inner statistics*.

**Exercise 2.** Make a violin plot of 1,000 samples from a standard normal. Describe where it is widest and how the median/quartile marks sit relative to the body.

### Intermediate

**Exercise 3.** Build a violin plot of the exponential distribution (mean 5, 1,000 samples). The KDE body will show some width *below zero* even though waiting times cannot be negative. Why does that happen, and which earlier concept explains it?

**Exercise 4.** Half-violin ("raincloud") variant: draw the mirrored KDE only on the left side and scatter the raw jittered data on the right. Implement it for the newspaper sample with `fill_betweenx`. What does this variant add over a standard violin?

### Advanced

**Exercise 5.** Violin bodies in matplotlib are scaled so each violin has the same maximum *width*, not the same *area* — which can mislead when comparing group sizes. Create two groups of very different sizes (50 vs. 5,000 samples from the same distribution), plot them side by side, and explain what the equal widths do and don't mean.

**Exercise 6.** Grand-finale dashboard: for the bimodal dataset, produce one figure with four panels — histogram, box plot, KDE, violin — and write (in a print statement) one sentence per panel on what it shows and what it misses. This is the summary of the whole visualization arc of this lesson.

## 7. Solutions

In [ ]:
# Exercise 1: reading the skew
print("Exercise 1:")
print("Shape clue: the body bulges low (around $18-30k) and stretches into a long")
print("thin neck toward $75k -- mass piled left, tail right.")
print("Statistics clue: the median (27.8) sits in the lower part of the Q1-Q3 box,")
print("and the mean (35.0) lies ABOVE the median -- the tail pulls it up.")

In [ ]:
# Exercise 2: violin of a standard normal
rng = np.random.default_rng(0)
z = rng.standard_normal(1000)

fig, ax = plt.subplots(figsize=(5, 5))
parts = ax.violinplot(z, showmedians=True)
parts["bodies"][0].set_alpha(0.5)
ax.set_title("standard normal: symmetric violin")
ax.set_xticks([])
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("Exercise 2: widest at 0 (the mode = mean = median), tapering symmetrically;")
print("the median mark cuts the body exactly in the middle -- no skew anywhere.")

In [ ]:
# Exercise 3: the boundary leak
rng = np.random.default_rng(0)
waits = rng.exponential(5, size=1000)

fig, ax = plt.subplots(figsize=(5, 5))
parts = ax.violinplot(waits, showmedians=True)
parts["bodies"][0].set_alpha(0.5)
ax.axhline(0, color="tab:red", linestyle="--", label="hard boundary: waits >= 0")
ax.set_xticks([])
ax.grid(True, alpha=0.3, axis="y")
ax.legend()
plt.tight_layout()
plt.show()

print("Exercise 3: the KDE puts a Gaussian mountain on every point, and mountains")
print("near 0 spill their left half below the boundary -- the kernel smears mass")
print("where no data can exist. Same bandwidth trade-off as the KDE notebook.")

In [ ]:
# Exercise 4: half-violin with raw data ("raincloud")
def gaussian(x, center, sigma):
    return np.exp(-0.5 * ((x - center) / sigma)**2) / (sigma * np.sqrt(2 * np.pi))

def kde(x, data, sigma):
    return np.mean([gaussian(x, xi, sigma) for xi in data], axis=0)

rng = np.random.default_rng(0)
grid = np.linspace(-10, 100, 500)
density = kde(grid, newspaper, 6)
scaled = density / density.max() * 0.4

fig, ax = plt.subplots(figsize=(6, 5))
ax.fill_betweenx(grid, -scaled, 0, color="tab:blue", alpha=0.5, label="KDE (half violin)")
ax.plot(rng.uniform(0.05, 0.3, len(newspaper)), newspaper, "o",
        color="tab:orange", label="raw data (jittered)")
ax.set_title("raincloud: half violin + the actual points")
ax.set_xticks([])
ax.set_ylabel("budget ($1000s)")
ax.grid(True, alpha=0.3, axis="y")
ax.legend()
plt.tight_layout()
plt.show()

print("Exercise 4: the mirrored half of a violin is redundant; spending that side")
print("on the raw points shows sample size and exact values the smooth body hides.")

In [ ]:
# Exercise 5: equal widths despite unequal n
rng = np.random.default_rng(0)
small = rng.normal(50, 10, size=50)
large = rng.normal(50, 10, size=5000)

fig, ax = plt.subplots(figsize=(7, 5))
parts = ax.violinplot([small, large], showmedians=True)
for body in parts["bodies"]:
    body.set_alpha(0.5)
ax.set_xticks([1, 2])
ax.set_xticklabels(["n = 50", "n = 5000"])
ax.grid(True, alpha=0.3, axis="y")
ax.set_title("Same max width, wildly different sample sizes")
plt.tight_layout()
plt.show()

print("Exercise 5: widths encode RELATIVE density within each violin, not counts;")
print("both bodies are drawn to the same maximum width, so nothing in the figure")
print("says one group has 100x the data. Overlay points or state n explicitly.")

In [ ]:
# Exercise 6: the four-panel finale
rng = np.random.default_rng(0)
bimodal = np.concatenate([rng.normal(30, 4, size=500), rng.normal(70, 4, size=500)])
grid = np.linspace(5, 95, 500)
sigma_b = 1.06 * bimodal.std() * len(bimodal)**(-1/5)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].hist(bimodal, bins=50, color="tab:blue", alpha=0.7)
axes[0].set_title("histogram")
axes[1].boxplot(bimodal, widths=0.5)
axes[1].set_title("box plot")
axes[1].set_xticks([])
axes[2].plot(grid, kde(grid, bimodal, sigma_b), color="tab:red", linewidth=2)
axes[2].set_title("KDE")
parts = axes[3].violinplot(bimodal, showmedians=True)
parts["bodies"][0].set_alpha(0.5)
axes[3].set_title("violin")
axes[3].set_xticks([])
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.suptitle("One dataset, four lenses")
plt.tight_layout()
plt.show()

print("Exercise 6:")
print("histogram: shows both clusters and counts, but shape depends on bins.")
print("box plot: robust five-number summary, completely blind to the two modes.")
print("KDE: smooth shape with both bumps, but no quartiles/median/outlier info.")
print("violin: shape AND summary in one -- at the cost of a bandwidth choice.")

## 8. Conclusion

- A **violin plot** = mirrored **KDE** body + **box-plot** statistics (median, mean, quartiles, whiskers) drawn inside — the information of both tools in one figure.
- Read the body's width as estimated density: bulges are clusters, a stretched thin section is a tail/skew; read the inner marks as the usual robust summary.
- Violins shine when **comparing groups** and when the shape matters (bimodality that box plots hide); their weaknesses are inherited from the KDE (bandwidth choice, boundary smearing) plus the equal-width scaling caveat.
- The full visualization arc of this lesson: **quantiles** (the numbers) → **box plot** (five numbers as a picture) → **KDE** (the whole estimated shape) → **violin** (shape + numbers together).

This concludes Lesson 3's toolkit for describing distributions — from expected value and variance through skewness, kurtosis, and the plots that show them. Next in the course: probability distributions with **two or more variables** (joint, marginal, conditional distributions, and covariance).